## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [ ]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,744 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,024 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [ ]:
!ollama pull bge-m3

In [ ]:
!pip install -U langchain langchain-core langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.6
    Uninstalling langchain-1.3.6:
      Successfully uninstalled langchain-1.3.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are 

In [ ]:
!pip install -U langchain-ollama

## Embedding Text with Ollama

Now that Ollama is running and the `bge-m3` model is pulled, we can use `langchain-ollama` to create embeddings for your text data.

First, we'll load your JSON file. Please ensure `direct_taxes_law_v1.json` is available in your environment, for example, by uploading it to your Colab session or mounting Google Drive.

In [ ]:
import json

# Load the JSON data from the file
try:
    with open('direct_taxes_law_v1.json', 'r') as f:
        data = json.load(f)
    print(f"Successfully loaded {len(data)} items from direct_taxes_law_v1.json")
except FileNotFoundError:
    print("Error: 'direct_taxes_law_v1.json' not found. Please upload the file or provide the correct path.")
    data = [] # Initialize data as empty list to prevent further errors
except json.JSONDecodeError:
    print("Error: Could not decode JSON from 'direct_taxes_law_v1.json'. Please check file content.")
    data = []

Successfully loaded 1204 items from direct_taxes_law_v1.json


Next, we initialize the `OllamaEmbeddings` using the `bge-m3` model. Note that the `base_url` is set to the default Ollama API address.

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings

# Initialize Ollama embeddings with the bge-m3 model
embeddings = OllamaEmbeddings(model="bge-m3", base_url="http://localhost:11434")
print("OllamaEmbeddings initialized with model: bge-m3")

OllamaEmbeddings initialized with model: bge-m3


In [ ]:
!pip install tqdm

In [ ]:
import tqdm

output = []
errors = []
with tqdm.tqdm(total=len(data), desc='Embedding laws') as pbar:
    for item in data:
        try:
          vector = embeddings.embed_query(item["finalSectionText"])
        except ValueError as e:
          print(f"Error embedding text: {e}")
          errors.append(item["sectionId"])
          continue
        output.append(
            {
              "id": item["sectionId"],
              "vector": vector,
              "text": item["finalSectionText"],
              "metadata": {k: v for k, v in item.items() if k != "finalSectionText"}
            }
        )
        pbar.update(1)
print("errors:", errors)

Embedding laws:  72%|███████▏  | 871/1204 [05:20<02:56,  1.88it/s]

Error embedding text: Error raised by inference API HTTP code: 500, {"error":"the input length exceeds the context length"}


Embedding laws: 100%|█████████▉| 1198/1204 [07:22<00:03,  1.80it/s]

Error embedding text: Error raised by inference API HTTP code: 500, {"error":"the input length exceeds the context length"}


Embedding laws: 100%|█████████▉| 1202/1204 [07:24<00:00,  2.70it/s]


In [ ]:
with open('output.json', 'w') as f:
    json.dump(output, f, indent=2)